- This code runs 5 independent base queries for seaching 3 languages and the keyword android once in the topic and once in the name, desc and readme
- All 5 queries includes three common qualifier: stars:>50, fork:false and archived:false
- Layer 1: Live Search - GitHub API
- Layer 2: there is no layer two in this version
- This is the final version used to create the list for step 2: AndroidManifest.xml check


In [3]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, set_key
from base64 import b64decode
from datetime import datetime, timedelta
from time import sleep, time as now

# === Load environment and tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7) if os.getenv(f"GITHUB_TOKEN_{i}")]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

# === Token rotation state ===
current_token_index = 0
last_used_token_index = -1
RATE_LIMIT_THRESHOLD = 1

# === Constants ===
START_DATE = os.getenv("START_DATE", "2008-01-01T00:00:00Z")
END_DATE = "2024-12-31T23:59:59Z"
INITIAL_WINDOW_HOURS = float(os.getenv("SEARCH_WINDOW_HOURS", 24))
MIN_WINDOW_HOURS = 1
MAX_WINDOW_HOURS = 90 * 24
MAX_RESULTS_PER_QUERY = 1000
TARGET_FILL_RATIO = 0.25

# === Output path ===
OUTPUT_DIR = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "step1_url_lookup_output.csv")

# === GitHub request headers ===
def get_headers():
    global current_token_index, last_used_token_index
    if last_used_token_index != current_token_index:
        print(f"🔄 Switching to token #{current_token_index + 1} / {len(tokens)}")
        last_used_token_index = current_token_index
    return {
        "Authorization": f"token {tokens[current_token_index]}",
        "Accept": "application/vnd.github.mercy-preview+json"
    }

def rotate_token():
    global current_token_index
    current_token_index = (current_token_index + 1) % len(tokens)
    print(f"🔁 Rotating to next token: #{current_token_index + 1}")
    if current_token_index == 0:
        print("⏳ All tokens used. Sleeping for 60 seconds to reset rate limits...")
        sleep(60)
        print("🔄 Resuming after sleep...")


def check_rate_limit():
    url = "https://api.github.com/rate_limit"
    res = requests.get(url, headers=get_headers())
    if res.status_code == 200:
        remaining = res.json()['rate']['remaining']
        #print(f"📉 Token #{current_token_index + 1} remaining requests: {remaining}")
        if remaining <= RATE_LIMIT_THRESHOLD:
            print("⚠️ Nearing rate limit. Rotating token...")
            rotate_token()
            return check_rate_limit()
    elif res.status_code == 403:
        print("🚫 Forbidden: Possibly rate-limited. Rotating token...")
        rotate_token()
        return check_rate_limit()


def get_readme(repo_full_name):
    url = f"https://api.github.com/repos/{repo_full_name}/readme"
    check_rate_limit()
    res = requests.get(url, headers=get_headers())
    if res.status_code == 200:
        content = res.json().get("content", "")
        return b64decode(content).decode("utf-8", errors="ignore")
    else:
        print(f"📄 README not found or inaccessible for {repo_full_name}")
    return ""


def get_topics(repo_full_name):
    url = f"https://api.github.com/repos/{repo_full_name}/topics"
    check_rate_limit()
    res = requests.get(url, headers=get_headers())
    if res.status_code == 200:
        return res.json().get("names", [])
    return []

def search_android_keyword(name, desc, topics, readme):
    fields = [name or "", desc or "", " ".join(topics), readme or ""]
    return "yes" if any("android" in f.lower() for f in fields) else "no"

def fetch_metadata(repo):
    full_name = repo.get("full_name")
    print(f"🔎 Fetching metadata for {full_name}")

    # Call only to enrich metadata (assume fork, archived, stars already valid)
    url = f"https://api.github.com/repos/{full_name}"
    check_rate_limit()
    res = requests.get(url, headers=get_headers())
    if res.status_code == 200:
        repo_data = res.json()
        lang = repo_data.get("language", "")
        readme = get_readme(full_name)
        topics = get_topics(full_name)
        ak = search_android_keyword(repo_data.get("name"), repo_data.get("description"), topics, readme)
        valid = "yes" if lang in ["Java", "Kotlin", "Dart"] or ak == "yes" else "no"
        return {
            "html_url": repo_data.get("html_url"),
            "name": repo_data.get("name"),
            "language": lang,
            "stars": repo_data.get("stargazers_count", 0),  # optional, still useful
            "created_at": repo_data.get("created_at", ""),
            "Android_keyword": ak,
            "Valid_Repo_Step1": valid
        }
    else:
        print(f"❌ Error fetching metadata for {full_name}: {res.status_code} - {res.text}")
    return None



def save_start_date(new_date_str):
    set_key("All_Tokens.env", "START_DATE", new_date_str)

# === Main Search Function ===
def search():
    base_queries = [
        "stars:>50 language:Kotlin fork:false archived:false",
        "stars:>50 language:Java fork:false archived:false",
        "stars:>50 language:Dart fork:false archived:false",
        "stars:>50 topic:android fork:false archived:false",
        "stars:>50 android in:name,description,readme fork:false archived:false",
    ]

    # Load last known start date from env
    start_date_str = os.getenv("START_DATE", "2008-01-01T00:00:00Z")
    base_start = datetime.strptime(start_date_str, "%Y-%m-%dT%H:%M:%SZ")
    end = datetime.strptime(END_DATE, "%Y-%m-%dT%H:%M:%SZ")

    if os.path.exists(OUTPUT_FILE):
        df_all = pd.read_csv(OUTPUT_FILE)
    else:
        df_all = pd.DataFrame()

    for q in base_queries:
        print(f"\n🔁 Starting new sweep for query: {q}")
        start = base_start
        window = timedelta(hours=INITIAL_WINDOW_HOURS)

        while start < end:
            since = start.strftime("%Y-%m-%dT%H:%M:%SZ")
            until_dt = min(start + window, end)
            until = until_dt.strftime("%Y-%m-%dT%H:%M:%SZ")
            print(f"\n🔍 Query window for [{q}]: {since} → {until} (window = {window})")

            q_base = f"{q} created:{since}..{until}"
            count_url = f"https://api.github.com/search/repositories?q={q_base}&per_page=1"
            check_rate_limit()
            count_res = requests.get(count_url, headers=get_headers())
            total_count = count_res.json().get("total_count", 0) if count_res.status_code == 200 else -1
            print(f"🔢 Estimated total: {total_count}")

            current_hours = window.total_seconds() / 3600
            if total_count >= MAX_RESULTS_PER_QUERY and current_hours > MIN_WINDOW_HOURS:
                window = timedelta(hours=max(current_hours // 2, MIN_WINDOW_HOURS))
                print(f"⚠️ Too many results. Shrinking window to {window}.")
                continue
            elif 0 <= total_count < MAX_RESULTS_PER_QUERY * TARGET_FILL_RATIO and current_hours * 2 <= MAX_WINDOW_HOURS:
                window = timedelta(hours=min(current_hours * 2, MAX_WINDOW_HOURS))
                print(f"📈 Too few results. Expanding window to {window}.")

            batch = []
            for page in range(1, 11):
                paged_url = f"https://api.github.com/search/repositories?q={q_base}&per_page=100&page={page}&sort=stars&order=desc"
                print(f"📄 Fetching page {page} for [{q}]...")
                check_rate_limit()
                res = requests.get(paged_url, headers=get_headers())
                if res.status_code == 200:
                    items = res.json().get("items", [])
                    if not items:
                        print("📭 No more items.")
                        break
                    print(f"✅ Page {page}: Retrieved {len(items)} repos")
                    for repo in items:
                        meta = fetch_metadata(repo)
                        if meta:
                            batch.append(meta)
                        sleep(1)
                elif res.status_code == 403:
                    print("⏳ Rate limited. Rotating token...")
                    rotate_token()
                    continue
                else:
                    print(f"❌ Error on page {page}: {res.status_code} - {res.text}")
                    break

            if len(batch) >= MAX_RESULTS_PER_QUERY:
                print(f"⚠️ Hit GitHub's 1,000 result cap. Shrinking window from {window}.")
                current_hours = window.total_seconds() / 3600
                window = timedelta(hours=max(current_hours // 2, MIN_WINDOW_HOURS))
                continue

            if batch:
                df_batch = pd.DataFrame(batch)
                df_all = pd.concat([df_all, df_batch], ignore_index=True)
                df_all.drop_duplicates(subset="html_url", inplace=True)
                df_all.to_csv(OUTPUT_FILE, index=False)
                print(f"💾 Saved {len(df_batch)} new repos to CSV")

            # Move to next time window
            start = until_dt + timedelta(seconds=1)

        print(f"✅ Finished full time sweep for query: {q}")

    # Update START_DATE to the end time (so future runs skip all covered ranges)
    save_start_date(end.strftime("%Y-%m-%dT%H:%M:%SZ"))
    print("\n🎉 Search complete for all queries. Final results saved.")

if __name__ == "__main__":
    search()



🔁 Starting new sweep for query: stars:>50 language:Kotlin fork:false archived:false

🔍 Query window for [stars:>50 language:Kotlin fork:false archived:false]: 2019-04-01T00:00:00Z → 2019-04-02T00:00:00Z (window = 1 day, 0:00:00)
🔄 Switching to token #1 / 6
🔢 Estimated total: 2
📈 Too few results. Expanding window to 2 days, 0:00:00.
📄 Fetching page 1 for [stars:>50 language:Kotlin fork:false archived:false]...
✅ Page 1: Retrieved 2 repos
🔎 Fetching metadata for tadfisher/gradle2nix
🔎 Fetching metadata for tfcporciuncula/dagger-journey
📄 Fetching page 2 for [stars:>50 language:Kotlin fork:false archived:false]...
📭 No more items.
💾 Saved 2 new repos to CSV

🔍 Query window for [stars:>50 language:Kotlin fork:false archived:false]: 2019-04-02T00:00:01Z → 2019-04-04T00:00:01Z (window = 2 days, 0:00:00)
🔢 Estimated total: 9
📈 Too few results. Expanding window to 4 days, 0:00:00.
📄 Fetching page 1 for [stars:>50 language:Kotlin fork:false archived:false]...
✅ Page 1: Retrieved 9 repos
🔎 Fetc

KeyboardInterrupt: 